In [38]:
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from roc_helper import plot_roc
from sklearn.metrics import roc_curve, auc

In [39]:
training_data = pd.read_csv("./data/train.csv")
test_data = pd.read_csv("./data/test.csv")

In [40]:
X_train = training_data
y_train = training_data['Transported']
X_submission_test = test_data
# from sklearn.model_selection import train_test_split

# X_training_data, X_test_data, y_training_data, y_test_data = train_test_split(
#    X_train, y_train, test_size=0.2, random_state=42
#)

## Data handling

Extract passenger group from PassengerId.

In [41]:
# Extract group information from PassengerId
if 'PassengerId' in X_train.columns:
    X_train['Group_num'] = X_train['PassengerId'].astype(str).str.split('_').str[0]
    X_train['Group_ID'] = X_train['PassengerId'].astype(str).str.split('_').str[1]
    X_train['Group_size'] = X_train.groupby('Group_num')['Group_num'].transform('count')
else:
    print('PassengerId not found')
X_train[['Group_num','Group_ID','Group_size']].head()

,Group_num,Group_ID,Group_size
0,0001,01,1
1,0002,01,1
2,0003,01,2
3,0003,02,2
4,0004,01,1


In [42]:
# Extract group information from PassengerId
if 'PassengerId' in X_submission_test.columns:
    X_submission_test['Group_num'] = X_submission_test['PassengerId'].astype(str).str.split('_').str[0]
    X_submission_test['Group_ID'] = X_submission_test['PassengerId'].astype(str).str.split('_').str[1]
    X_submission_test['Group_size'] = X_submission_test.groupby('Group_num')['Group_num'].transform('count')
else:
    print('PassengerId not found')
X_submission_test[['Group_num','Group_ID','Group_size']].head()

,Group_num,Group_ID,Group_size
0,0013,01,1
1,0018,01,1
2,0019,01,1
3,0021,01,1
4,0023,01,1


Ensure GroupKFold cross validation using passenger groupp as grouping variable

In [43]:
from sklearn.model_selection import GroupKFold
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)
print('Prepared GroupKFold with', n_splits, 'folds')

Prepared GroupKFold with 5 folds


Use 5 folds

Track mean and std CV accuracy

## Feature engineering

### Cabin
Split into Deck, CabinNumber, Side
Create Deck frequency encoding (out of fold)
Create relative cabin number within deck
Add CabinMissing Flag

In [44]:
# Cabin processing: split and create basic cabin features
if 'Cabin' in X_train.columns:
    X_train[['Deck','Cabin_num','Side']] = X_train['Cabin'].astype(str).str.split('/', expand=True)
    X_train['Deck'] = X_train['Deck'].replace('nan','Unknown').fillna('Unknown')
    X_train['CabinMissing'] = X_train['Cabin'].isnull()
else:
    print('Cabin column not found')
# numeric cabin and relative cabin number per deck
X_train['Cabin_num'] = pd.to_numeric(X_train['Cabin_num'], errors='coerce')
X_train['Cabin_num_rel'] = X_train.groupby('Deck')['Cabin_num'].transform(lambda s: (s - s.min())/(s.max()-s.min()) if s.notnull().any() and s.max()!=s.min() else 0)
# Deck frequency encoding (simple)
deck_freq = X_train['Deck'].fillna('Unknown').value_counts(normalize=True).to_dict()
X_train['Deck_freq'] = X_train['Deck'].map(deck_freq).fillna(0)
X_train[['Deck','Cabin_num','Cabin_num_rel','CabinMissing','Deck_freq']].head()

,Deck,Cabin_num,Cabin_num_rel,CabinMissing,Deck_freq
0,B,0.0,0.000000,False,0.089612
1,F,0.0,0.000000,False,0.321408
2,A,0.0,0.000000,False,0.029449
3,A,0.0,0.000000,False,0.029449
4,F,1.0,0.000528,False,0.321408


In [45]:
# Cabin processing: split and create basic cabin features
if 'Cabin' in X_train.columns:
    X_submission_test[['Deck','Cabin_num','Side']] = X_submission_test['Cabin'].astype(str).str.split('/', expand=True)
    X_submission_test['Deck'] = X_submission_test['Deck'].replace('nan','Unknown').fillna('Unknown')
    X_submission_test['CabinMissing'] = X_submission_test['Cabin'].isnull()
else:
    print('Cabin column not found')
# numeric cabin and relative cabin number per deck
X_submission_test['Cabin_num'] = pd.to_numeric(X_submission_test['Cabin_num'], errors='coerce')
X_submission_test['Cabin_num_rel'] = X_submission_test.groupby('Deck')['Cabin_num'].transform(lambda s: (s - s.min())/(s.max()-s.min()) if s.notnull().any() and s.max()!=s.min() else 0)
# Deck frequency encoding (simple)
deck_freq = X_submission_test['Deck'].fillna('Unknown').value_counts(normalize=True).to_dict()
X_submission_test['Deck_freq'] = X_submission_test['Deck'].map(deck_freq).fillna(0)
X_submission_test[['Deck','Cabin_num','Cabin_num_rel','CabinMissing','Deck_freq']].head()

,Deck,Cabin_num,Cabin_num_rel,CabinMissing,Deck_freq
0,G,3.0,0.000000,False,0.285714
1,F,4.0,0.000000,False,0.337854
2,C,0.0,0.000000,False,0.083002
3,C,1.0,0.002933,False,0.083002
4,F,5.0,0.000530,False,0.337854


### Spending
Create TotalSpent
Create log1p TotalSpent
Create spend properties for Spa, VRDeck, RoomService, FoodCourt, ShoppingMall
Create ZeroSpend flas
Create HighSpend percentile flag

In [46]:
# Spending features
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for c in spend_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].fillna(0)
# Total and log total
X_train['TotalSpent'] = X_train[[c for c in spend_cols if c in X_train.columns]].sum(axis=1)
X_train['LogTotalSpent'] = np.log1p(X_train['TotalSpent'])
X_train['ZeroSpend'] = (X_train['TotalSpent']==0).astype(int)
X_train['HighSpend'] = (X_train['TotalSpent'] > X_train['TotalSpent'].quantile(0.95)).astype(int)
X_train[['TotalSpent','LogTotalSpent','ZeroSpend','HighSpend']].head()

,TotalSpent,LogTotalSpent,ZeroSpend,HighSpend
0,0.0,0.000000,1,0
1,736.0,6.602588,0,0
2,10383.0,9.248021,0,1
3,5176.0,8.551981,0,0
4,1091.0,6.995766,0,0


In [47]:
# Spending features
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for c in spend_cols:
    if c in X_submission_test.columns:
        X_submission_test[c] = X_submission_test[c].fillna(0)
# Total and log total
X_submission_test['TotalSpent'] = X_submission_test[[c for c in spend_cols if c in X_submission_test.columns]].sum(axis=1)
X_submission_test['LogTotalSpent'] = np.log1p(X_submission_test['TotalSpent'])
X_submission_test['ZeroSpend'] = (X_submission_test['TotalSpent']==0).astype(int)
X_submission_test['HighSpend'] = (X_submission_test['TotalSpent'] > X_submission_test['TotalSpent'].quantile(0.95)).astype(int)
X_submission_test[['TotalSpent','LogTotalSpent','ZeroSpend','HighSpend']].head()

,TotalSpent,LogTotalSpent,ZeroSpend,HighSpend
0,0.0,0.000000,1,0
1,2832.0,7.949091,0,0
2,0.0,0.000000,1,0
3,7418.0,8.911800,0,1
4,645.0,6.470800,0,0


### CryoSleep
if missing and TotalSpend == 0 set True
if missing and TotalSpend != 0 set False
add inconsistency flag (CryoSleep True but spend > 0) 

In [48]:
# CryoSleep imputation based on spending
if 'CryoSleep' in X_train.columns:
    X_train['CryoSleep'] = X_train['CryoSleep'].map({True:1, False:0}).astype('float')
    X_train.loc[X_train['CryoSleep'].isnull() & (X_train['TotalSpent']==0), 'CryoSleep'] = 1
    X_train.loc[X_train['CryoSleep'].isnull() & (X_train['TotalSpent']>0), 'CryoSleep'] = 0
    X_train['CryoSleep'] = X_train['CryoSleep'].astype(int)
    X_train['CryoSleep_inconsistent'] = ((X_train['CryoSleep']==1) & (X_train['TotalSpent']>0)).astype(int)
else:
    print('CryoSleep column not found')
X_train[['CryoSleep','CryoSleep_inconsistent']].head()

,CryoSleep,CryoSleep_inconsistent
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [49]:
# CryoSleep imputation based on spending
if 'CryoSleep' in X_submission_test.columns:
    X_submission_test['CryoSleep'] = X_submission_test['CryoSleep'].map({True:1, False:0}).astype('float')
    X_submission_test.loc[X_submission_test['CryoSleep'].isnull() & (X_submission_test['TotalSpent']==0), 'CryoSleep'] = 1
    X_submission_test.loc[X_submission_test['CryoSleep'].isnull() & (X_submission_test['TotalSpent']>0), 'CryoSleep'] = 0
    X_submission_test['CryoSleep'] = X_submission_test['CryoSleep'].astype(int)
    X_submission_test['CryoSleep_inconsistent'] = ((X_submission_test['CryoSleep']==1) & (X_submission_test['TotalSpent']>0)).astype(int)
else:
    print('CryoSleep column not found')
X_submission_test[['CryoSleep','CryoSleep_inconsistent']].head()

,CryoSleep,CryoSleep_inconsistent
0,1,0
1,0,0
2,1,0
3,0,0
4,0,0


### Group features
Group size
Group mean TotalSpend (out of fold)
Group CryoSleep ratio (out of fold)
Individual spend mins group mean
Flag if group all zero spend

In [50]:
# Group-level features (out-of-fold) using GroupKFold
def group_agg_oof(df, group_col, agg_col, n_splits=5, aggfunc='mean'):
    oof = pd.Series(index=df.index, dtype=float)
    gkf_local = GroupKFold(n_splits=n_splits)
    for tr_idx, val_idx in gkf_local.split(df, groups=df[group_col]):
        train = df.iloc[tr_idx]
        val = df.iloc[val_idx]
        stats = train.groupby(group_col)[agg_col].agg(aggfunc)
        oof.iloc[val_idx] = val[group_col].map(stats).fillna(train[agg_col].agg(aggfunc))
    return oof.fillna(df[agg_col].agg(aggfunc))

if 'Group_num' in X_train.columns:
    X_train['Group_mean_spend'] = group_agg_oof(X_train, 'Group_num', 'TotalSpent', n_splits=n_splits, aggfunc='mean')
    group_sum_oof = group_agg_oof(X_train, 'Group_num', 'TotalSpent', n_splits=n_splits, aggfunc='sum')
    X_train['Group_zero_spend'] = (group_sum_oof == 0).astype(int)
    X_train['Group_cryo_ratio'] = group_agg_oof(X_train, 'Group_num', 'CryoSleep', n_splits=n_splits, aggfunc='mean')
    X_train['Spend_minus_group_mean'] = X_train['TotalSpent'] - X_train['Group_mean_spend']
else:
    print('No Group_num found')
X_train[['Group_mean_spend','Group_zero_spend','Group_cryo_ratio','Spend_minus_group_mean']].head()

,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean
0,1453.141645,0,0.360800,-1453.141645
1,1444.678648,0,0.359454,-708.678648
2,1424.740725,0,0.361806,8958.259275
3,1424.740725,0,0.361806,3751.259275
4,1424.740725,0,0.361806,-333.740725


In [51]:
# Group-level features for test: map from full training stats
if 'Group_num' in X_submission_test.columns:
    grp_mean = X_train.groupby('Group_num')['TotalSpent'].mean()
    grp_sum = X_train.groupby('Group_num')['TotalSpent'].sum()
    grp_cryo = X_train.groupby('Group_num')['CryoSleep'].mean()
    X_submission_test['Group_mean_spend'] = X_submission_test['Group_num'].map(grp_mean).fillna(X_train['TotalSpent'].mean())
    X_submission_test['Group_zero_spend'] = X_submission_test['Group_num'].map(grp_sum).fillna(0).eq(0).astype(int)
    X_submission_test['Group_cryo_ratio'] = X_submission_test['Group_num'].map(grp_cryo).fillna(X_train['CryoSleep'].mean())
    X_submission_test['Spend_minus_group_mean'] = X_submission_test['TotalSpent'] - X_submission_test['Group_mean_spend']
else:
    print('No Group_num found')
X_submission_test[['Group_mean_spend','Group_zero_spend','Group_cryo_ratio','Spend_minus_group_mean']].head()

,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean
0,1440.866329,1,0.360635,-1440.866329
1,1440.866329,1,0.360635,1391.133671
2,1440.866329,1,0.360635,-1440.866329
3,1440.866329,1,0.360635,5977.133671
4,1440.866329,1,0.360635,-795.866329


### Name
Extract last name
Compute surname frequency
Compute out of fold target encoding for surname

In [52]:
# Extract surname and frequency
if 'Name' in X_train.columns:
    X_train['Surname'] = X_train['Name'].astype(str).str.split().str[-1]
    X_train['Surname_freq'] = X_train['Surname'].map(X_train['Surname'].value_counts())
else:
    print('Name column missing')
X_train[['Name','Surname','Surname_freq']].head()

,Name,Surname,Surname_freq
0,Maham Ofracculy,Ofracculy,1
1,Juanna Vines,Vines,4
2,Altark Susent,Susent,6
3,Solam Susent,Susent,6
4,Willy Santantines,Santantines,6


In [53]:
# Extract surname and frequency
if 'Name' in X_submission_test.columns:
    X_submission_test['Surname'] = X_submission_test['Name'].astype(str).str.split().str[-1]
    X_submission_test['Surname_freq'] = X_submission_test['Surname'].map(X_submission_test['Surname'].value_counts())
else:
    print('Name column missing')
X_submission_test[['Name','Surname','Surname_freq']].head()

,Name,Surname,Surname_freq
0,Nelly Carsoning,Carsoning,4
1,Lerome Peckers,Peckers,1
2,Sabih Unhearfus,Unhearfus,1
3,Meratz Caltilter,Caltilter,1
4,Brence Harperez,Harperez,3


### Categorical Encoding
Use K fold target encoding with smoothing and noise for:
    HomePlanet
    Destination
    Deck
    Surname
    Group ID
Ensure no leaking: encoding must be done only on training folds

In [54]:
# Target encoding (out-of-fold) helper
from sklearn.model_selection import KFold
def target_encode_oof(df, col, target, n_splits=5, seed=42, smoothing=1.0):
    oof = pd.Series(index=df.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = df[target].mean()
    for tr_idx, val_idx in kf.split(df):
        train, val = df.iloc[tr_idx], df.iloc[val_idx]
        stats = train.groupby(col)[target].agg(['mean','count'])
        # smoothing based on count
        stats['smooth'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
        mapping = stats['smooth'].to_dict()
        oof.iloc[val_idx] = val[col].map(mapping).fillna(global_mean)
    return oof.fillna(global_mean)
# Apply to columns present
for c in ['HomePlanet','Destination','Deck','Surname','Group_ID']:
    if c in X_train.columns:
        X_train[c + '_te'] = target_encode_oof(pd.concat([X_train, y_train.rename('target')], axis=1), c, 'target', n_splits=5)
X_train.filter(regex='_te$').head()

,HomePlanet_te,Destination_te,Deck_te,Surname_te,Group_ID_te
0,0.655610,0.470533,0.722140,0.503624,0.476034
1,0.425299,0.473258,0.445917,0.834541,0.477605
2,0.657524,0.469041,0.497702,0.875906,0.474391
3,0.657524,0.469041,0.497702,0.875906,0.559157
4,0.420591,0.469041,0.440913,0.417271,0.474391


Missing values:
VIP missing -> False
Age impute using median per HomePlanet
Cabin missing treated as category

In [55]:
# Missing value handling
if 'VIP' in X_train.columns:
    X_train['VIP'] = X_train['VIP'].fillna(False).astype(int)
if 'Age' in X_train.columns and 'HomePlanet' in X_train.columns:
    X_train['Age'] = X_train.groupby('HomePlanet')['Age'].transform(lambda s: s.fillna(s.median()))
if 'Age' in X_train.columns:
    X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
for c in ['Deck','Side','Cabin_num']:
    if c in X_train.columns:
        X_train[c] = X_train[c].fillna('Unknown')
X_train[['VIP','Age','Deck','Side']].head()

C:\Users\sbrad\AppData\Local\Temp\ipykernel_18320\403975049.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train['VIP'] = X_train['VIP'].fillna(False).astype(int)


,VIP,Age,Deck,Side
0,0,39.0,B,P
1,0,24.0,F,S
2,1,58.0,A,S
3,0,33.0,A,S
4,0,16.0,F,S


In [56]:
# Missing value handling
if 'VIP' in X_submission_test.columns:
    X_submission_test['VIP'] = X_submission_test['VIP'].fillna(False).astype(int)
if 'Age' in X_submission_test.columns and 'HomePlanet' in X_submission_test.columns:
    X_submission_test['Age'] = X_submission_test.groupby('HomePlanet')['Age'].transform(lambda s: s.fillna(s.median()))
if 'Age' in X_submission_test.columns:
    X_submission_test['Age'] = X_submission_test['Age'].fillna(X_submission_test['Age'].median())
for c in ['Deck','Side','Cabin_num']:
    if c in X_submission_test.columns:
        X_submission_test[c] = X_submission_test[c].fillna('Unknown')
X_submission_test[['VIP','Age','Deck','Side']].head()

C:\Users\sbrad\AppData\Local\Temp\ipykernel_18320\2973202822.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_submission_test['VIP'] = X_submission_test['VIP'].fillna(False).astype(int)


,VIP,Age,Deck,Side
0,0,27.0,G,S
1,0,19.0,F,S
2,0,31.0,C,S
3,0,38.0,C,S
4,0,20.0,F,S


## Modeling

In [57]:
# Build feature list: drop identifiers and original target
drop_cols = ['PassengerId','Name','Cabin'] if 'PassengerId' in X_train.columns else ['Name','Cabin']
features = [c for c in X_train.columns if c not in drop_cols + ['Transported','target'] and not c.startswith('Deck_')]
# include engineered features and TE columns
features = [c for c in X_train.columns if c not in drop_cols + ['Transported','target'] and not c.startswith('Deck_')]
print('feature count:', len(features))
features[:50]

feature count: 33


['HomePlanet',
 'CryoSleep',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Group_num',
 'Group_ID',
 'Group_size',
 'Deck',
 'Cabin_num',
 'Side',
 'CabinMissing',
 'Cabin_num_rel',
 'TotalSpent',
 'LogTotalSpent',
 'ZeroSpend',
 'HighSpend',
 'CryoSleep_inconsistent',
 'Group_mean_spend',
 'Group_zero_spend',
 'Group_cryo_ratio',
 'Spend_minus_group_mean',
 'Surname',
 'Surname_freq',
 'HomePlanet_te',
 'Destination_te',
 'Surname_te',
 'Group_ID_te']

In [58]:
# Build feature list: drop identifiers and original target
drop_cols = ['PassengerId','Name','Cabin'] if 'PassengerId' in X_submission_test.columns else ['Name','Cabin']
features = [c for c in X_submission_test.columns if c not in drop_cols + ['Transported','target'] and not c.startswith('Deck_')]
# include engineered features and TE columns
features = [c for c in X_submission_test.columns if c not in drop_cols + ['Transported','target'] and not c.startswith('Deck_')]
print('feature count:', len(features))
features[:50]

feature count: 29


['HomePlanet',
 'CryoSleep',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Group_num',
 'Group_ID',
 'Group_size',
 'Deck',
 'Cabin_num',
 'Side',
 'CabinMissing',
 'Cabin_num_rel',
 'TotalSpent',
 'LogTotalSpent',
 'ZeroSpend',
 'HighSpend',
 'CryoSleep_inconsistent',
 'Group_mean_spend',
 'Group_zero_spend',
 'Group_cryo_ratio',
 'Spend_minus_group_mean',
 'Surname',
 'Surname_freq']

Use XGBClassifier with:
    tree_method = hist
    eval_metric = logloss
    learning_rate small (0.02 - 0.05)
    max_depth (4-8)
    subsample and colsample_bytree tuned
    strong regularization
    early stopping on validation fold

In [59]:
# XGBoost CV example using GroupKFold (use xgb.train with DMatrix for reliable early stopping)
from sklearn.metrics import roc_auc_score
params = dict(tree_method='hist', learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=2.0)
oof = np.zeros(len(X_train))
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'] if 'Group_num' in X_train.columns else None)):
    X_tr, X_val = X_train[features].iloc[tr_idx].copy(), X_train[features].iloc[val_idx].copy()
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    # encode object/category columns to integer codes using train mapping (avoid DMatrix categorical issues)
    cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
    for col in cat_cols:
        train_cats = pd.Categorical(X_tr[col].astype(str))
        mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
        X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
        X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
    # fallback: factorize any remaining object columns
    for col in X_tr.select_dtypes(include=['object']).columns:
        X_tr[col], _ = pd.factorize(X_tr[col])
        X_val[col], _ = pd.factorize(X_val[col])
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic','eval_metric':'logloss'})
    bst = xgb.train(params_train, dtrain, num_boost_round=1000, evals=[(dval,'valid')], early_stopping_rounds=50, verbose_eval=False)
    # get best ntree limit if available
    best_ntree = getattr(bst, 'best_ntree_limit', None)
    if best_ntree is None:
        best_ntree = getattr(bst, 'best_iteration', None)
    if best_ntree:
        preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
    else:
        preds = bst.predict(dval)
    oof[val_idx] = preds
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    print(f'Fold {fold} AUC:', fold_auc)
print('CV mean AUC:', np.mean(fold_scores), 'std:', np.std(fold_scores))

Fold 0 AUC: 0.8904791010141901
Fold 1 AUC: 0.8909511778944723
Fold 2 AUC: 0.9026007375462951
Fold 3 AUC: 0.8988280417078884
Fold 4 AUC: 0.8904152140048959
CV mean AUC: 0.8946548544335483 std: 0.005092764238756722


Train with:
    5 fold GroupKFold
    multiple random seeds (5-10)
    Store out of fold predictions

In [ ]:
# Example: average across multiple seeds using xgb.train for reliable early stopping
from sklearn.metrics import roc_auc_score
seeds = [42, 7, 2021, 345,21321 ,321]
oof_all = np.zeros((len(X_train), len(seeds)))
for i, s in enumerate(seeds):
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic', 'eval_metric':'logloss', 'random_state': s})
    oof_tmp = np.zeros(len(X_train))
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'] if 'Group_num' in X_train.columns else None)):
        X_tr, X_val = X_train[features].iloc[tr_idx].copy(), X_train[features].iloc[val_idx].copy()
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        # convert object dtype columns to pandas 'category' so XGBoost can handle them as categorical
        # encode object/category columns to integer codes using train mapping (avoid DMatrix categorical issues)
        cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
        for col in cat_cols:
            train_cats = pd.Categorical(X_tr[col].astype(str))
            mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
            X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
            X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
        # fallback: factorize any remaining object columns
        for col in X_tr.select_dtypes(include=['object']).columns:
            X_tr[col], _ = pd.factorize(X_tr[col])
            X_val[col], _ = pd.factorize(X_val[col])
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(params_train, dtrain, num_boost_round=1000, evals=[(dval, 'valid')], early_stopping_rounds=30, verbose_eval=False)
        best_ntree = getattr(bst, 'best_ntree_limit', None) or getattr(bst, 'best_iteration', None)
        if best_ntree:
            preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
        else:
            preds = bst.predict(dval)
        oof_tmp[val_idx] = preds
    oof_all[:, i] = oof_tmp

# average across seeds
oof_avg = oof_all.mean(axis=1)
print('Seeds OOF AUC:', roc_auc_score(y_train, oof_avg))

## Ensembling

In [ ]:
# Ensemble: simple average across seed OOFs (oof_all produced earlier)
try:
    ensemble_oof = oof_all.mean(axis=1)
    print('Ensemble OOF AUC:', roc_auc_score(y_train, ensemble_oof))
except NameError:
    print('oof_all not found; run CV cells first')

Ensemble OOF AUC: 0.8934698775664904


Average predictions across:
    All folds
    All seeds
    Optimize seed weights using CV

## Output

Print fold scores
Print overall CV mean and std
roc curves for all

In [ ]:
training_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean,Surname,Surname_freq,HomePlanet_te,Destination_te,Deck_te,Surname_te,Group_ID_te
0,0001_01,Europa,0,B/0/P,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,...,1,0.0,0.0,Ofracculy,1,0.655610,0.470533,0.722140,0.503624,0.476034
1,0002_01,Earth,0,F/0/S,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,...,0,0.0,0.0,Vines,4,0.425299,0.473258,0.445917,0.834541,0.477605
2,0003_01,Europa,0,A/0/S,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,...,0,0.0,2603.5,Susent,6,0.657524,0.469041,0.497702,0.875906,0.474391
3,0003_02,Europa,0,A/0/S,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,...,0,0.0,-2603.5,Susent,6,0.657524,0.469041,0.497702,0.875906,0.559157
4,0004_01,Earth,0,F/1/S,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,...,0,0.0,0.0,Santantines,6,0.420591,0.469041,0.440913,0.417271,0.474391


In [ ]:
test_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,LogTotalSpent,ZeroSpend,HighSpend,CryoSleep_inconsistent,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean,Surname,Surname_freq
0,0013_01,Earth,1,G/3/S,TRAPPIST-1e,27.0,0,0.0,0.0,0.0,...,0.000000,1,0,0,0.0,1,1.0,0.0,Carsoning,4
1,0018_01,Earth,0,F/4/S,TRAPPIST-1e,19.0,0,0.0,9.0,0.0,...,7.949091,0,0,0,2832.0,0,0.0,0.0,Peckers,1
2,0019_01,Europa,1,C/0/S,55 Cancri e,31.0,0,0.0,0.0,0.0,...,0.000000,1,0,0,0.0,1,1.0,0.0,Unhearfus,1
3,0021_01,Europa,0,C/1/S,TRAPPIST-1e,38.0,0,0.0,6652.0,0.0,...,8.911800,0,1,0,7418.0,0,0.0,0.0,Caltilter,1
4,0023_01,Earth,0,F/5/S,TRAPPIST-1e,20.0,0,10.0,0.0,635.0,...,6.470800,0,0,0,645.0,0,0.0,0.0,Harperez,3


In [ ]:
# before loop
test_preds_seeds = np.zeros((len(X_submission_test), len(seeds)))

for i, s in enumerate(seeds):
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic', 'eval_metric':'logloss', 'random_state': s})
    oof_tmp = np.zeros(len(X_train))
    test_preds_folds = np.zeros((len(X_submission_test), gkf.get_n_splits()))
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'])):
        X_tr = X_train[features].iloc[tr_idx].copy()
        X_val = X_train[features].iloc[val_idx].copy()
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        # create mapping from this fold's train and apply to val and test
        cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
        for col in cat_cols:
            train_cats = pd.Categorical(X_tr[col].astype(str))
            mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
            X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
            X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
        # prepare test copy and apply same mapping
        X_test_enc = X_submission_test[features].copy()
        for col in cat_cols:
            X_test_enc[col] = X_test_enc[col].astype(str).map(mapping).astype(float).fillna(-1)

        # fallback factorize if any object remains (rare)
        for col in X_tr.select_dtypes(include=['object']).columns:
            X_tr[col], _ = pd.factorize(X_tr[col])
            X_val[col], _ = pd.factorize(X_val[col])
            X_test_enc[col], _ = pd.factorize(X_test_enc[col])

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(params_train, dtrain, num_boost_round=1000,
                        evals=[(dval, 'valid')], early_stopping_rounds=30, verbose_eval=False)

        best_ntree = getattr(bst, 'best_ntree_limit', None) or getattr(bst, 'best_iteration', None)
        if best_ntree:
            preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
            test_pred = bst.predict(xgb.DMatrix(X_test_enc), iteration_range=(0, int(best_ntree)))
        else:
            preds = bst.predict(dval)
            test_pred = bst.predict(xgb.DMatrix(X_test_enc))

        oof_tmp[val_idx] = preds
        test_preds_folds[:, fold] = test_pred

    oof_all[:, i] = oof_tmp
    # average across folds for this seed and store
    test_preds_seeds[:, i] = test_preds_folds.mean(axis=1)

# final test prediction: average across seeds
final_test_preds = test_preds_seeds.mean(axis=1)
# boolean label
submission = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': final_test_preds > 0.5})
submission.to_csv('submission.csv', index=False)